# AML Graph Intelligence Engine — Notebook 2
## Phase 2 (Graph Construction) + Phase 3 (Structural Feature Engineering)

**What this notebook does, in one sentence:** it turns the flat transaction table into a *graph* (accounts = nodes, transfers = edges) and then measures, for every account, a set of *structural* numbers that describe how it sits inside the money-flow network — the numbers Phase 4 will use to detect laundering and Phase 5 will turn into a risk score.

**Why we build the graph now, before any detection.** A transaction table can only answer "who paid whom, once." A graph answers "where does money *flow*, across many hops, over time." Money laundering is invisible row-by-row and only becomes visible as a *shape* in the network (a fan, a chain, a cycle). You cannot detect a shape you have not built. This notebook builds the canvas; Notebook 3 finds the shapes.

**Confirmed schema:**

| Role | Real column | File |
|---|---|---|
| Sender (node) | `Sender_account` | transactions.csv |
| Receiver (node) | `Receiver_account` | transactions.csv |
| Amount (edge weight) | `amount_local_npr` | transactions.csv |
| Timestamp | `Date` + `Time` | transactions.csv |
| Account key (for KYC) | `account_id` | accounts.csv |
| Label (validation only) | `is_suspicious_tx` | ml_features.csv |

**Key facts from EDA that shaped the choices below:** 100,222 transactions, 65,339 accounts, sparse graph (avg degree ~3), data clean (no missing/dupes/self-loops/orphans), label positive rate 0.335%, and only ~3,557 accounts both send *and* receive (the "pass-through" accounts where layering can happen).

## Step 0 — Setup

`networkx` is the graph library (build the network, run centrality/components). `pandas`/`numpy` for tables and math.

> **Concept — what is a graph?** A *graph* is just "dots and arrows." Each **node** (dot) is an account. Each **edge** (arrow) is a transfer, pointing **from** sender **to** receiver. Because direction matters in money flow (sending NPR 10M is the opposite of receiving it), we use a **directed** graph (a "DiGraph"). That single idea — keeping the arrow's direction — is what lets us reason about *flow* later.

In [ ]:
import os, pickle, time, warnings
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 170)

print("networkx:", nx.__version__)
print("pandas  :", pd.__version__)

networkx: 3.6.1
pandas  : 2.2.2


## Step 1 — Load the data and re-attach the label

We mount Drive, load the three files we need, rebuild the combined timestamp, and re-attach the transaction label using the **positional alignment** the EDA already proved is safe (the files are row-for-row aligned at 100% on `Sender/Receiver/Date/Time`).

We do **not** need `graph_edges.csv` here, it's just a 6-column subset of `transactions.csv` at the same granularity, so `transactions.csv` already contains everything `graph_edges.csv` has, plus more.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR   = "/content/drive/MyDrive/STUDENT_DATASET"
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- confirmed schema constants ----
SENDER_COL     = "Sender_account"
RECEIVER_COL   = "Receiver_account"
AMOUNT_COL     = "amount_local_npr"      # NPR-normalized -> currencies comparable
DATE_COL       = "Date"
TIME_COL       = "Time"
ACCOUNT_ID_COL = "account_id"
LABEL_COL      = "is_suspicious_tx"

# ---- load ----
df_txn      = pd.read_csv(os.path.join(DATA_DIR, "transactions.csv"))
df_accounts = pd.read_csv(os.path.join(DATA_DIR, "accounts.csv"))
df_features = pd.read_csv(os.path.join(DATA_DIR, "ml_features.csv"))

# ---- combined timestamp ----
df_txn["ts"] = pd.to_datetime(df_txn[DATE_COL].astype(str) + " " + df_txn[TIME_COL].astype(str),
                              errors="coerce")

# ---- re-attach label by positional alignment (EDA proved 100% row match) ----
assert len(df_txn) == len(df_features), "Row counts differ - positional join unsafe!"
key_cols = [SENDER_COL, RECEIVER_COL, DATE_COL, TIME_COL]
match = (df_txn[key_cols].reset_index(drop=True) ==
         df_features[key_cols].reset_index(drop=True)).all(axis=1).mean()
print(f"Row-alignment check on {key_cols}: {match:.4%}")
assert match > 0.999, "Files not aligned - stop and investigate before trusting the label."
df_txn[LABEL_COL] = df_features[LABEL_COL].values

print(f"\ntransactions : {len(df_txn):,} rows")
print(f"accounts     : {len(df_accounts):,} rows")
print(f"suspicious tx: {df_txn[LABEL_COL].sum():,}  ({df_txn[LABEL_COL].mean():.3%})")
print(f"time span    : {df_txn['ts'].min()}  ->  {df_txn['ts'].max()}")

Mounted at /content/drive
Row-alignment check on ['Sender_account', 'Receiver_account', 'Date', 'Time']: 100.0000%

transactions : 100,222 rows
accounts     : 65,339 rows
suspicious tx: 336  (0.335%)
time span    : 2022-10-07 10:35:19  ->  2022-11-06 21:04:35


## Step 2 — Build the **multigraph** (one edge per transaction)

> **Concept — why a *multi*graph?** Two accounts can transact many times. If A pays B on Monday and again on Friday, those are two *different* events with two timestamps. A normal graph allows only one edge per pair and would force us to throw one away. A **MultiDiGraph** keeps *every* transaction as its own edge.

Each edge carries three attributes: `amount` (NPR), `ts` (when), and `is_susp` (the label, carried along purely so we can *validate* later — it is never used to build features).

In [ ]:
t0 = time.time()
MG = nx.MultiDiGraph()

# add every account as a node first (so isolated/receive-only accounts still exist)
MG.add_nodes_from(df_accounts[ACCOUNT_ID_COL].tolist())

# add one edge per transaction
edge_records = df_txn[[SENDER_COL, RECEIVER_COL, AMOUNT_COL, "ts", LABEL_COL]].itertuples(index=False)
MG.add_edges_from(
    (s, r, {"amount": a, "ts": t, "is_susp": int(lab)})
    for s, r, a, t, lab in edge_records
)

print(f"MultiDiGraph built in {time.time()-t0:.1f}s")
print(f"  nodes: {MG.number_of_nodes():,}")
print(f"  edges: {MG.number_of_edges():,}   (should equal transaction count: {len(df_txn):,})")

MultiDiGraph built in 1.5s
  nodes: 65,339
  edges: 100,222   (should equal transaction count: 100,222)


## Step 3 — Build the **aggregated graph** (one edge per account pair)

> **Concept — why also build a *collapsed* graph?** Algorithms like PageRank, betweenness, connected components and cycle-finding don't care that A paid B fifty times, they care that "a relationship A→B exists, and here is its total strength." Running them on the multigraph is slower and noisier. So we collapse all A→B transactions into **one** edge whose weight is the *total* NPR that flowed A→B, plus we keep the transaction count and the first/last time for context.

We maintain two synchronized views of the network:

- a temporal multigraph for time-sensitive typologies and

- an aggregated weighted digraph for structural centrality

and apply each algorithm to the view it's suited to.

In [ ]:
t0 = time.time()
agg = (df_txn
       .groupby([SENDER_COL, RECEIVER_COL])
       .agg(total_amount=(AMOUNT_COL, "sum"),
            txn_count   =(AMOUNT_COL, "size"),
            mean_amount =(AMOUNT_COL, "mean"),
            first_ts    =("ts", "min"),
            last_ts     =("ts", "max"),
            susp_txns   =(LABEL_COL, "sum"))
       .reset_index())

G = nx.DiGraph()
G.add_nodes_from(df_accounts[ACCOUNT_ID_COL].tolist())
for row in agg.itertuples(index=False):
    G.add_edge(row.Sender_account, row.Receiver_account,
               weight=row.total_amount, txn_count=row.txn_count,
               mean_amount=row.mean_amount, first_ts=row.first_ts,
               last_ts=row.last_ts, susp_txns=row.susp_txns)

print(f"Aggregated DiGraph built in {time.time()-t0:.1f}s")
print(f"  nodes: {G.number_of_nodes():,}")
print(f"  unique directed account-pairs (edges): {G.number_of_edges():,}")
print(f"  (multigraph had {MG.number_of_edges():,} edges -> "
      f"avg {MG.number_of_edges()/max(G.number_of_edges(),1):.2f} transactions per pair)")

Aggregated DiGraph built in 0.7s
  nodes: 65,339
  unique directed account-pairs (edges): 50,586
  (multigraph had 100,222 edges -> avg 1.98 transactions per pair)


## Step 4 — Attach KYC attributes to the nodes

> **Concept — node attributes.** A node isn't just an ID, it's an account with *properties*: is it a Politically Exposed Person (PEP)? On a sanctions list? How old is the account? These come from `accounts.csv`. They are **not** structural (they say nothing about the graph shape), but in Phase 5 we blend them into the hybrid score as the **"KYC risk"** component. We attach them now so everything about an account lives in one place.

We also compute `account_age_days` (older accounts are generally lower-risk; brand-new accounts that immediately move large sums are a classic mule signature) relative to the last day in the dataset.

In [ ]:
REF_DATE = df_txn["ts"].max().normalize()   # reference "today" = last day of data

acc = df_accounts.copy()
acc["opened_dt"] = pd.to_datetime(acc["opened"], errors="coerce")
acc["account_age_days"] = (REF_DATE - acc["opened_dt"]).dt.days

# map of account_id -> dict of KYC attributes
kyc_cols = ["pep_flag", "sanctions_hit", "risk_grade", "acct_type",
            "is_person", "city", "institution", "account_age_days"]
kyc_map = acc.set_index(ACCOUNT_ID_COL)[kyc_cols].to_dict("index")

# attach to BOTH graphs (cheap; keeps them self-describing)
nx.set_node_attributes(G,  kyc_map)
nx.set_node_attributes(MG, kyc_map)

# quick confirm
sample_node = next(iter(G.nodes))
print("Example node attributes:")
print(f"  account {sample_node}: {G.nodes[sample_node]}")

Example node attributes:
  account 8724731955: {'pep_flag': 0, 'sanctions_hit': 0, 'risk_grade': 'RISK-LOW', 'acct_type': 'FIXED', 'is_person': True, 'city': 'Hetauda', 'institution': 'HBL', 'account_age_days': 2506}


## Step 5 — Sanity-check the graph

In [ ]:
print("GRAPH SANITY CHECK")
print("=" * 50)
print(f"Nodes (= accounts)        : {G.number_of_nodes():,}   (EDA expected 65,339)")
print(f"Edges (= unique pairs)    : {G.number_of_edges():,}")
print(f"Transactions (multi-edges): {MG.number_of_edges():,}   (EDA expected 100,222)")
self_loops = nx.number_of_selfloops(G)
print(f"Self-loops                : {self_loops}   (EDA expected 0)")
density = nx.density(G)
print(f"Density                   : {density:.6f}   (tiny -> sparse graph, as expected)")
print(f"Is weakly connected?      : {nx.is_weakly_connected(G)}")

GRAPH SANITY CHECK
Nodes (= accounts)        : 65,339   (EDA expected 65,339)
Edges (= unique pairs)    : 50,586
Transactions (multi-edges): 100,222   (EDA expected 100,222)
Self-loops                : 0   (EDA expected 0)
Density                   : 0.000012   (tiny -> sparse graph, as expected)
Is weakly connected?      : False


---
## Phase 3 begins — Structural feature engineering

From here on, every cell adds columns to a per-account feature table. We'll assemble them all at the end. We compute **flow features** straight from the transaction table (fastest, least error-prone) and **topological features** from the graph (where networkx shines).

## Step 6 — Degree & flow features

> **Concepts:**
> - **out-degree / in-degree** = how many *distinct* accounts you send to / receive from. A sender to 1 account behaves very differently from a sender to 40.
> - **weighted in/out** = total NPR received / sent. Volume, not just count.
> - **flow ratio = total_out / total_in.** This is one of the single most powerful AML features. A normal business *accumulates* (in > out) or *spends* (out > in). A **mule / pass-through** account receives money and immediately forwards almost all of it, so its ratio sits suspiciously close to **1.0** — money flows *through* it, it doesn't *keep* any. We'll lean on this heavily.
> - **unique counterparties** = breadth of the account's relationships (a smurf collector receives from *many* unrelated senders).

In [ ]:
# --- from the transaction table (vectorized, fast) ---
out_amt = df_txn.groupby(SENDER_COL)[AMOUNT_COL].agg(weighted_out="sum", txn_out_count="size")
in_amt  = df_txn.groupby(RECEIVER_COL)[AMOUNT_COL].agg(weighted_in ="sum", txn_in_count ="size")
uniq_out = df_txn.groupby(SENDER_COL)[RECEIVER_COL].nunique().rename("out_degree")
uniq_in  = df_txn.groupby(RECEIVER_COL)[SENDER_COL].nunique().rename("in_degree")

feat = pd.DataFrame(index=df_accounts[ACCOUNT_ID_COL].values)
feat.index.name = ACCOUNT_ID_COL
for s in [out_amt, in_amt, uniq_out, uniq_in]:
    feat = feat.join(s)

feat = feat.fillna(0)
feat["weighted_total"] = feat["weighted_in"] + feat["weighted_out"]
feat["net_flow"]       = feat["weighted_in"] - feat["weighted_out"]
# flow ratio: out / in. Use a small epsilon so receive-only accounts don't divide by zero.
feat["flow_ratio"]     = feat["weighted_out"] / (feat["weighted_in"] + 1.0)
# "pass-through-ness": close to 1 AND meaningful volume both directions
feat["is_pass_through"] = ((feat["in_degree"] > 0) & (feat["out_degree"] > 0)).astype(int)

print(f"Pass-through accounts (both send & receive): {int(feat['is_pass_through'].sum()):,}"
      f"   (EDA expected ~3,557)")
print("\nFlow-ratio summary for pass-through accounts only:")
print(feat.loc[feat['is_pass_through']==1, 'flow_ratio'].describe().round(3))

Pass-through accounts (both send & receive): 3,557   (EDA expected ~3,557)

Flow-ratio summary for pass-through accounts only:
count     3557.000
mean        23.688
std        501.097
min          0.000
25%          0.452
50%          1.096
75%          4.260
max      27511.555
Name: flow_ratio, dtype: float64


## Step 7 — Reciprocity

> **Concept — reciprocity.** If A→B *and* B→A both exist, the edge is **reciprocal** (money goes both ways). Ordinary commerce is often one-directional (you pay the shop; the shop doesn't pay you back). High reciprocity can mean either legitimate two-way business *or* the back-and-forth churn used to manufacture fake transaction history. We record, per account, what fraction of its outgoing partners also send back to it.

In [ ]:
recip_frac = {}
for n in G.nodes:
    succ = set(G.successors(n))
    if not succ:
        recip_frac[n] = 0.0
        continue
    back = sum(1 for m in succ if G.has_edge(m, n))
    recip_frac[n] = back / len(succ)

feat["reciprocity_frac"] = pd.Series(recip_frac)
feat["reciprocity_frac"] = feat["reciprocity_frac"].fillna(0)
print("Accounts with any reciprocal relationship:",
      int((feat['reciprocity_frac'] > 0).sum()))
print(feat["reciprocity_frac"].describe().round(3))

Accounts with any reciprocal relationship: 1493
count    65339.000
mean         0.020
std          0.133
min          0.000
25%          0.000
50%          0.000
75%          0.000
max          1.000
Name: reciprocity_frac, dtype: float64


## Step 8 — Connected components (where cycles live)

> **Concepts:**
> - **Weakly Connected Component (WCC):** ignore arrow direction — which accounts are reachable from each other at all? The graph likely has one giant WCC plus many tiny islands.
> - **Strongly Connected Component (SCC):** *respecting* direction — a set of accounts where you can get from any one to any other *and back*, following the arrows. **This is the key one for laundering:** a cycle (money leaving an account and eventually returning to it) can *only* exist inside an SCC of size ≥ 2. So the size of the SCC an account belongs to is a cheap, powerful proxy for "is this account part of a circular-flow structure?" Most legitimate accounts sit in an SCC of size 1 (themselves). Accounts in a large SCC are immediately interesting.

In [ ]:
# strongly connected components
scc_sets = list(nx.strongly_connected_components(G))
scc_size_map = {n: len(c) for c in scc_sets for n in c}
feat["scc_size"] = pd.Series(scc_size_map)

# weakly connected components
wcc_sets = list(nx.weakly_connected_components(G))
wcc_size_map = {n: len(c) for c in wcc_sets for n in c}
feat["wcc_size"] = pd.Series(wcc_size_map)

feat["in_cycle_scc"] = (feat["scc_size"] >= 2).astype(int)

big_sccs = sorted((len(c) for c in scc_sets), reverse=True)[:10]
print(f"Total SCCs: {len(scc_sets):,}")
print(f"Largest SCC sizes (top 10): {big_sccs}")
print(f"Accounts inside an SCC of size >= 2 (can be in a cycle): "
      f"{int(feat['in_cycle_scc'].sum()):,}")
print(f"Largest WCC size: {max(len(c) for c in wcc_sets):,} "
      f"({max(len(c) for c in wcc_sets)/G.number_of_nodes():.1%} of all accounts)")

Total SCCs: 64,273
Largest SCC sizes (top 10): [7, 7, 7, 7, 6, 6, 6, 6, 6, 6]
Accounts inside an SCC of size >= 2 (can be in a cycle): 1,525
Largest WCC size: 43 (0.1% of all accounts)


## Step 9 — Centrality: PageRank, clustering, (approx) betweenness

> **Concepts:**
> - **PageRank** (the Google one): a node is "important" if important nodes point *to* it. In money terms, an account that *receives* from many accounts that themselves receive a lot is a natural **collector** — exactly the endpoint of a fan-in / smurfing structure. PageRank surfaces collectors far better than raw in-degree.
> - **Clustering coefficient:** of neighbours, how many are also connected to each other? Tight little clusters (high clustering) can indicate a coordinated ring rather than independent counterparties.
> - **Betweenness:** how often a node sits *on the shortest path* between other nodes — i.e. it's a **bridge** that money must pass through. Pass-through "layering" hubs score high. Exact betweenness on 65k nodes is slow, so we use networkx's built-in **k-sample approximation** (accurate enough for ranking, runs in seconds).

In [ ]:
t0 = time.time()

# PageRank on the aggregated, weighted graph
pr = nx.pagerank(G, weight="weight")
feat["pagerank"] = pd.Series(pr)
print(f"PageRank done in {time.time()-t0:.1f}s")

# Clustering coefficient (treat as undirected for the local-density notion)
t0 = time.time()
clust = nx.clustering(G.to_undirected())
feat["clustering"] = pd.Series(clust)
print(f"Clustering done in {time.time()-t0:.1f}s")

# Approximate betweenness via k-sampling (k shortest-path sources)
t0 = time.time()
k = min(800, G.number_of_nodes())
btw = nx.betweenness_centrality(G, k=k, normalized=True, seed=42)
feat["betweenness_approx"] = pd.Series(btw)
print(f"Approx betweenness (k={k}) done in {time.time()-t0:.1f}s")

feat[["pagerank", "clustering", "betweenness_approx"]] = \
    feat[["pagerank", "clustering", "betweenness_approx"]].fillna(0)
print("\nTop 5 accounts by PageRank (likely collectors):")
print(feat.sort_values("pagerank", ascending=False)
        [["in_degree","out_degree","weighted_in","pagerank"]].head())

PageRank done in 0.6s
Clustering done in 3.4s
Approx betweenness (k=800) done in 87.4s

Top 5 accounts by PageRank (likely collectors):
            in_degree  out_degree   weighted_in  pagerank
account_id                                               
9683990807       21.0         3.0  5.788601e+08   0.00021
4255313773       21.0         2.0  6.548921e+08   0.00021
6086421020       21.0        21.0  4.581836e+08   0.00021
2670254274       21.0         3.0  3.976148e+08   0.00021
5251195293       20.0         1.0  5.863960e+08   0.00020


## Step 10 — Attach KYC + the validation label, then save everything

We merge the KYC fields onto the feature table, then attach the **account-level** label. Remember the raw label is per-*transaction*; an account is marked `account_suspicious = 1` if it was the sender or receiver of **any** flagged transaction. **This column is for validation/scoring only — it is never an input to a structural feature.** We keep it in the file so Notebook 5 can measure precision@k without re-deriving it.

We also flag the obvious injected synthetic ring (12-digit `9000000000xx` IDs) in a column `injected_ring` — again **validation only**.

Then we pickle both graphs and write `node_features.csv`. Every later notebook starts by loading these three files.

In [ ]:
# --- KYC merge ---
feat = feat.join(acc.set_index(ACCOUNT_ID_COL)[kyc_cols])

# --- account-level label (validation only) ---
susp_senders   = set(df_txn.loc[df_txn[LABEL_COL]==1, SENDER_COL])
susp_receivers = set(df_txn.loc[df_txn[LABEL_COL]==1, RECEIVER_COL])
susp_accounts  = susp_senders | susp_receivers
feat["account_suspicious"] = feat.index.isin(susp_accounts).astype(int)

# --- injected synthetic ring marker (validation only; NEVER a feature) ---
feat["injected_ring"] = (feat.index >= 100_000_000_000).astype(int)

feat = feat.reset_index()

print(f"Feature table: {feat.shape[0]:,} accounts x {feat.shape[1]} columns")
print(f"account_suspicious = 1 : {int(feat['account_suspicious'].sum()):,} accounts "
      f"({feat['account_suspicious'].mean():.3%})")
print(f"injected_ring      = 1 : {int(feat['injected_ring'].sum()):,} accounts")

# --- SAVE ---
feat_path = os.path.join(OUTPUT_DIR, "node_features.csv")
feat.to_csv(feat_path, index=False)
with open(os.path.join(OUTPUT_DIR, "graph_multi.gpickle"), "wb") as f:
    pickle.dump(MG, f)
with open(os.path.join(OUTPUT_DIR, "graph_agg.gpickle"), "wb") as f:
    pickle.dump(G, f)

print("\nSaved:")
print(" ", feat_path)
print(" ", os.path.join(OUTPUT_DIR, "graph_multi.gpickle"))
print(" ", os.path.join(OUTPUT_DIR, "graph_agg.gpickle"))
print("\nColumns:", list(feat.columns))

Feature table: 65,339 accounts x 28 columns
account_suspicious = 1 : 319 accounts (0.488%)
injected_ring      = 1 : 171 accounts

Saved:
  /content/drive/MyDrive/aml-hackathon-data/outputs/node_features.csv
  /content/drive/MyDrive/aml-hackathon-data/outputs/graph_multi.gpickle
  /content/drive/MyDrive/aml-hackathon-data/outputs/graph_agg.gpickle

Columns: ['account_id', 'weighted_out', 'txn_out_count', 'weighted_in', 'txn_in_count', 'out_degree', 'in_degree', 'weighted_total', 'net_flow', 'flow_ratio', 'is_pass_through', 'reciprocity_frac', 'scc_size', 'wcc_size', 'in_cycle_scc', 'pagerank', 'clustering', 'betweenness_approx', 'pep_flag', 'sanctions_hit', 'risk_grade', 'acct_type', 'is_person', 'city', 'institution', 'account_age_days', 'account_suspicious', 'injected_ring']


## Step 11 — Smell test: do structural outliers overlap with real suspicious accounts?

This is **not** our detector yet. It's a quick gut-check that our features carry *signal* — because if even crude single-feature rankings already over-represent known-suspicious accounts. We rank accounts by three raw structural signals and measure what fraction of the top 200 are actually suspicious, versus the 0.3-ish% base rate.

> **How to read it:** the base rate of suspicious accounts is tiny. If "top 200 by PageRank" is, say, 10% suspicious, that's a **30×+ lift** over chance from a *single* feature — strong evidence the structural approach works. If a signal shows ~base-rate overlap, that feature alone is weak (still possibly useful in combination).

In [ ]:
base_rate = feat["account_suspicious"].mean()
print(f"Base rate of suspicious accounts: {base_rate:.3%}\n")

def topk_lift(col, k=200):
    top = feat.sort_values(col, ascending=False).head(k)
    hit = top["account_suspicious"].mean()
    ring = int(top["injected_ring"].sum())
    print(f"  top {k} by {col:<20}: {hit:6.2%} suspicious  "
          f"(lift {hit/base_rate:5.1f}x)   injected-ring caught: {ring}")

print("Single-feature lift over base rate (higher = more signal):")
for c in ["pagerank", "betweenness_approx", "weighted_out", "in_degree", "out_degree"]:
    topk_lift(c)

# flow-ratio is a 'closeness to 1' signal, only meaningful for pass-through accounts
pt = feat[feat["is_pass_through"]==1].copy()
pt["flow_closeness"] = -(pt["flow_ratio"] - 1.0).abs()   # higher = closer to 1.0
top = pt.sort_values("flow_closeness", ascending=False).head(200)
print(f"\n  top 200 pass-through accts by flow-ratio~1.0: "
      f"{top['account_suspicious'].mean():.2%} suspicious "
      f"(lift {top['account_suspicious'].mean()/base_rate:.1f}x)")

Base rate of suspicious accounts: 0.488%

Single-feature lift over base rate (higher = more signal):
  top 200 by pagerank            :  1.50% suspicious  (lift   3.1x)   injected-ring caught: 1
  top 200 by betweenness_approx  :  3.00% suspicious  (lift   6.1x)   injected-ring caught: 6
  top 200 by weighted_out        :  1.00% suspicious  (lift   2.0x)   injected-ring caught: 0
  top 200 by in_degree           :  5.00% suspicious  (lift  10.2x)   injected-ring caught: 8
  top 200 by out_degree          :  6.50% suspicious  (lift  13.3x)   injected-ring caught: 8

  top 200 pass-through accts by flow-ratio~1.0: 16.00% suspicious (lift 32.8x)
